In [9]:
import os
import pandas as pd
from dwave.system import LeapHybridCQMSampler
from dimod import ConstrainedQuadraticModel, BinaryQuadraticModel, QuadraticModel

def parse_inputs(data_file, capacity):
    df = pd.read_csv(data_file, names=['cost', 'weight'])
    if not capacity:
        capacity = int(0.8 * sum(df['weight']))
        print("\nSetting weight capacity to 80% of total: {}".format(str(capacity)))
    return df['cost'], df['weight'], capacity

def build_knapsack_cqm(costs, weights, max_weight):
    num_items = len(costs)
    print("\nBuilding a CQM for {} items.".format(str(num_items)))

    cqm = ConstrainedQuadraticModel()
    obj = BinaryQuadraticModel(vartype='BINARY')
    constraint = QuadraticModel()

    for i in range(num_items):
        obj.add_variable(i)
        obj.set_linear(i, -costs[i])
        constraint.add_variable('BINARY', i)
        constraint.set_linear(i, weights[i])

    cqm.set_objective(obj)
    cqm.add_constraint(constraint, sense="<=", rhs=max_weight, label='capacity')

    return cqm

def parse_solution(sampleset, costs, weights):
    feasible_sampleset = sampleset.filter(lambda row: row.is_feasible)

    if not len(feasible_sampleset):
        raise ValueError("No feasible solution found")

    best = feasible_sampleset.first

    selected_item_indices = [key for key, val in best.sample.items() if val==1.0]
    selected_weights = list(weights.loc[selected_item_indices])
    selected_costs = list(costs.loc[selected_item_indices])

    print("\nFound best solution at energy {}".format(best.energy))
    print("\nSelected item numbers (0-indexed):", selected_item_indices)
    print("\nSelected item weights: {}, total = {}".format(selected_weights, sum(selected_weights)))
    print("\nSelected item costs: {}, total = {}".format(selected_costs, sum(selected_costs)))

def main(filename, capacity):
    sampler = LeapHybridCQMSampler()

    costs, weights, capacity = parse_inputs(filename, capacity)

    cqm = build_knapsack_cqm(costs, weights, capacity)

    print("Submitting CQM to solver {}.".format(sampler.solver.name))
    sampleset = sampler.sample_cqm(cqm, label='Example - Knapsack')

    parse_solution(sampleset, costs, weights)

# Call the main function directly
main('data/small.csv', 1000)



Building a CQM for 7 items.
Submitting CQM to solver hybrid_constrained_quadratic_model_version1.

Found best solution at energy -405.0

Selected item numbers (0-indexed): [0, 1, 2, 3, 4, 5, 6]

Selected item weights: [12, 27, 11, 17, 20, 10, 15], total = 112

Selected item costs: [35, 85, 30, 50, 70, 80, 55], total = 405


In [10]:
import os
import pandas as pd
from dwave.system import LeapHybridCQMSampler
from dimod import ConstrainedQuadraticModel, BinaryQuadraticModel, QuadraticModel

def parse_inputs(data_file, capacity):
    df = pd.read_csv(data_file, names=['cost', 'weight'])
    if not capacity:
        capacity = int(0.8 * sum(df['weight']))
        print("\nSetting weight capacity to 80% of total: {}".format(str(capacity)))
    return df['cost'], df['weight'], capacity

def build_knapsack_cqm(costs, weights, max_weight):
    num_items = len(costs)
    print("\nBuilding a CQM for {} items.".format(str(num_items)))

    cqm = ConstrainedQuadraticModel()
    obj = BinaryQuadraticModel(vartype='BINARY')
    constraint = QuadraticModel()

    for i in range(num_items):
        obj.add_variable(i)
        obj.set_linear(i, -costs[i])
        constraint.add_variable('BINARY', i)
        constraint.set_linear(i, weights[i])

    cqm.set_objective(obj)
    cqm.add_constraint(constraint, sense="<=", rhs=max_weight, label='capacity')

    return cqm

def parse_solution(sampleset, costs, weights):
    feasible_sampleset = sampleset.filter(lambda row: row.is_feasible)

    if not len(feasible_sampleset):
        raise ValueError("No feasible solution found")

    best = feasible_sampleset.first

    selected_item_indices = [key for key, val in best.sample.items() if val==1.0]
    selected_weights = list(weights.loc[selected_item_indices])
    selected_costs = list(costs.loc[selected_item_indices])

    print("\nFound best solution at energy {}".format(best.energy))
    print("\nSelected item numbers (0-indexed):", selected_item_indices)
    print("\nSelected item weights: {}, total = {}".format(selected_weights, sum(selected_weights)))
    print("\nSelected item costs: {}, total = {}".format(selected_costs, sum(selected_costs)))

def main(filename, capacity):
    sampler = LeapHybridCQMSampler()

    costs, weights, capacity = parse_inputs(filename, capacity)

    cqm = build_knapsack_cqm(costs, weights, capacity)

    print("Submitting CQM to solver {}.".format(sampler.solver.name))
    sampleset = sampler.sample_cqm(cqm, label='Example - Knapsack')

    parse_solution(sampleset, costs, weights)

# Call the main function directly
main('data/large.csv', 1000)



Building a CQM for 68 items.
Submitting CQM to solver hybrid_constrained_quadratic_model_version1.

Found best solution at energy -2086.0

Selected item numbers (0-indexed): [3, 5, 9, 11, 12, 13, 14, 15, 16, 19, 20, 24, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 38, 44, 46, 54, 55, 60, 61, 62, 63, 66]

Selected item weights: [31, 1, 11, 26, 4, 34, 36, 22, 27, 40, 12, 24, 67, 56, 37, 15, 52, 74, 43, 71, 28, 20, 46, 63, 17, 19, 14, 33, 29, 10, 23, 13], total = 998

Selected item costs: [67, 91, 68, 53, 84, 85, 100, 43, 30, 54, 10, 49, 74, 57, 45, 18, 62, 79, 55, 86, 64, 78, 81, 72, 23, 96, 83, 58, 70, 92, 88, 71], total = 2086
